In [1]:
import polars as pl
import gc
from datetime import datetime
from catboost import CatBoostClassifier
from sklearn.metrics import average_precision_score, classification_report
from features import apply_type_casting, generate_features, cat_features

In [2]:
df = pl.scan_parquet('../data/train_features_v4.parquet')

df_train_lazy = df.filter(pl.col('event_dttm') < datetime(2025, 5, 1))

df_train_fraud = df_train_lazy.filter((pl.col('target') == 1)).collect()
df_train_nonfraud = df_train_lazy.filter((pl.col('target') == 0)).collect().sample(n=3_000_000, seed=42)

df_train = pl.concat([df_train_fraud, df_train_nonfraud]).sample(fraction=1.0, shuffle=True, seed=42)

del df_train_fraud, df_train_nonfraud, df_train_lazy
gc.collect()

df_val = df.filter(
    (pl.col('event_date').dt.year() == 2025) & 
    (pl.col('event_date').dt.month() == 5)
).collect()

In [3]:
drop_cols =['customer_id', 'event_id', 'event_dttm', 'event_date', 'target']

X_train = df_train.drop(drop_cols).to_pandas()
y_train = df_train.select('target').to_pandas()

X_val = df_val.drop(drop_cols).to_pandas()
y_val = df_val.select('target').to_pandas()

del df_train, df_val
gc.collect()

0

In [4]:
model = CatBoostClassifier(
    iterations=1500,
    learning_rate=0.03,
    depth=7,
    l2_leaf_reg=4.0,
    task_type='GPU',
    scale_pos_weight=68,
    eval_metric='PRAUC',
    metric_period=50,
    od_type='Iter',
    od_wait=50,
    random_seed=42
)

model.fit(X_train, y_train, cat_features=cat_features, eval_set=(X_val, y_val), verbose=50)
preds = model.predict_proba(X_val)[:, 1]
metric = average_precision_score(y_val, preds)

print(metric)

Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	learn: 0.8220233	test: 0.2501493	best: 0.2501493 (0)	total: 1.33s	remaining: 33m 16s
50:	learn: 0.8566266	test: 0.3271205	best: 0.3271205 (50)	total: 1m 1s	remaining: 29m 12s
100:	learn: 0.8685990	test: 0.3601362	best: 0.3601362 (100)	total: 2m 24s	remaining: 33m 16s
150:	learn: 0.8755741	test: 0.3769131	best: 0.3769131 (150)	total: 3m 57s	remaining: 35m 20s
200:	learn: 0.8797860	test: 0.3864162	best: 0.3864162 (200)	total: 5m 31s	remaining: 35m 44s
250:	learn: 0.8828829	test: 0.3929470	best: 0.3929470 (250)	total: 7m 12s	remaining: 35m 51s
300:	learn: 0.8852951	test: 0.3978661	best: 0.3978707 (299)	total: 8m 54s	remaining: 35m 27s
350:	learn: 0.8871904	test: 0.4016854	best: 0.4016854 (350)	total: 10m 31s	remaining: 34m 28s
400:	learn: 0.8888785	test: 0.4044128	best: 0.4044128 (400)	total: 12m 6s	remaining: 33m 11s
450:	learn: 0.8903588	test: 0.4071558	best: 0.4071558 (450)	total: 13m 41s	remaining: 31m 50s
500:	learn: 0.8917221	test: 0.4091428	best: 0.4091428 (500)	total: 15m 16s	r

In [5]:
model.get_feature_importance(prettified=True)

,Feature Id,Importances
0,client_op_seq_num,9.686434
1,amt_sum_24h,8.698794
2,mcc_code,7.924712
3,client_expanding_mean_amt,7.586967
4,amt_sum_1h,6.619501
5,amt_diff_from_exp_mean,6.412621
6,op_count_24h,5.393776
7,event_desc,5.292519
8,op_count_1h,5.265023
9,operaton_amt,5.131616


In [6]:
model.save_model('../models/catboost_v5.cbm')